# 第2回講義 宿題

## 課題
今回のLessonで学んだことを元に，MNISTのファッション版 (Fashion MNIST，クラス数10) をソフトマックス回帰によって分類してみましょう．

Fashion MNISTの詳細については以下のリンクを参考にしてください．

Fashion MNIST: https://github.com/zalandoresearch/fashion-mnist

### 目標値
Accuracy: 80%

### ルール
- 訓練データは`x_train`， `y_train`，テストデータは`x_test`で与えられます．
- 予測ラベルは one_hot表現ではなく0~9のクラスラベル で表してください．
- **下のセルで指定されている`x_train、y_train`以外の学習データは使わないでください．**
- **隠れ層のない単層のモデルとして実装してください.**
- **ソフトマックス回帰のアルゴリズム部分の実装はnumpyのみで行ってください** (sklearnやtensorflowなどは使用しないでください)．
    - データの前処理部分でsklearnの関数を使う (例えば `sklearn.model_selection.train_test_split`) のは問題ありません．

### 提出方法
- 2つのファイルを提出していただきます．
    1. テストデータ (`x_test`) に対する予測ラベルをcsv形式で保存し，**Omnicampusの宿題タブから「第2回 機械学習基礎」を選択して**提出してください．
    2. それに対応するpythonのコードを　ファイル＞ダウンロード＞.pyをダウンロード　から保存し，**Omnicampusの宿題タブから「第2回 機械学習基礎 (code)」を選択して**提出してください．pythonファイル自体の提出ではなく，「提出内容」の部分にコード全体をコピー&ペーストしてください．
      
- なお，採点は1で行い，2はコードの確認用として利用します（成績優秀者はコード内容を公開させていただくかもしれません）．コードの内容を変更した場合は，**1と2の両方を提出し直してください**．

### 評価方法
- 予測ラベルの`y_test`に対する精度 (Accuracy) で評価します．
- 即時採点しLeader Boardを更新します（採点スケジュールは別アナウンス）．
- 締切時の点数を最終的な評価とします．

### ドライブのマウント

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
# 作業ディレクトリを指定
#work_dir = 'drive/MyDrive/Colab Notebooks/DLBasics2026_colab'
# 作業ディレクトリを指定
work_dir = '/content/drive/MyDrive/■松尾研_DL基礎_2026_E資格_20261009-1225/2.松尾研_DL基礎_E資格_2_20261016/1.講義資料_松尾研_DL基礎_E資格_2_20261016'

### データの読み込み（このセルは修正しないでください）

In [26]:
import os
import sys

import numpy as np
import pandas as pd

sys.modules['tensorflow'] = None

def load_fashionmnist():
    # 学習データ
    x_train = np.load(work_dir + '/Lecture02/data/x_train.npy')
    y_train = np.load(work_dir + '/Lecture02/data/y_train.npy')

    # テストデータ
    x_test = np.load(work_dir + '/Lecture02/data/x_test.npy')

    x_train = x_train.reshape(-1, 784).astype('float32') / 255
    y_train = np.eye(10)[y_train.astype('int32')]
    x_test = x_test.reshape(-1, 784).astype('float32') / 255

    return x_train, y_train, x_test

### ソフトマックス回帰の実装

In [37]:
x_train, y_train, x_test = load_fashionmnist()

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

def softmax(x, axis=1):
    x = x - x.max(axis, keepdims=True)  # expのoverflowを防ぐ
    x_exp = np.exp(x)
    return x_exp / x_exp.sum(axis, keepdims=True)

# logの中身が0になるのを防ぐ
def np_log(x):
    return np.log(np.clip(a=x, a_min=1e-10, a_max=1e+10))

# 重み
W = np.random.uniform(low=-0.08, high=0.08, size=(784, 10)).astype('float32')  # 重み: (784, 10)
b = np.zeros(shape=(10,)).astype('float32')

# 学習データと検証データに分割
x_train, x_valid, y_train, y_valid = train_test_split(x_train, y_train, test_size=0.1)

def train(x, t, eps=1.0):
    """
    :param x: np.ndarray, 入力データ, (batch_size, 入力の次元数)
    :param t: np.ndarray, 教師ラベル, (batch_size, 出力の次元数)
    :param eps: float, 学習率
    """
    global W, b

    batch_size = x.shape[0]

    # 予測
    y_hat = softmax(np.matmul(x, W) + b)  # (batch_size, 出力の次元数)

    # 目的関数の評価
    cost = (- t * np_log(y_hat)).sum(axis=1).mean()
    delta = y_hat - t  # (batch_size, 出力の次元数)

    # パラメータの更新
    dW = np.matmul(x.T, delta) / batch_size  # (入力の次元数, 出力の次元数)
    db = np.matmul(np.ones(shape=(batch_size,)), delta) / batch_size  # (出力の次元数,)
    W -= eps * dW
    b -= eps * db

    return cost

def valid(x, t):
    y_hat = softmax(np.matmul(x, W) + b)
    cost = (- t * np_log(y_hat)).sum(axis=1).mean()

    return cost, y_hat

#batch_size = 100
#for epoch in range(20):
for epoch in range(200):
    cost = train(x_train, y_train, eps=0.1)   # ← ここだけ変更
    cost, y_pred = valid(x_valid, y_valid)
    if epoch % 10 == 9 or epoch == 0:
        print(...)

for epoch in range(100):
    # 学習データをシャッフルしてミニバッチ学習
    idx = np.random.permutation(len(x_train))
    for i in range(0, len(x_train), batch_size):
        batch = idx[i:i + batch_size]
        cost = train(x_train[batch], y_train[batch], eps=0.1)

    cost, y_pred = valid(x_valid, y_valid)
    print('EPOCH: {}, Valid Cost: {:.3f}, Valid Accuracy: {:.3f}'.format(
        epoch + 1,
        cost,
        accuracy_score(y_valid.argmax(axis=1), y_pred.argmax(axis=1))
    ))

#submission = pd.Series(y_pred.argmax(axis=1), name='label')
#submission.to_csv(work_dir + '/Lecture02/submission_pred_02.csv', header=True, index_label='id')

from google.colab import files
files.download(work_dir + '/Lecture02/submission_pred_02.csv')

EPOCH: 1, Valid Cost: 0.562, Valid Accuracy: 0.809
EPOCH: 2, Valid Cost: 0.504, Valid Accuracy: 0.828
EPOCH: 3, Valid Cost: 0.483, Valid Accuracy: 0.835
EPOCH: 4, Valid Cost: 0.482, Valid Accuracy: 0.836
EPOCH: 5, Valid Cost: 0.480, Valid Accuracy: 0.836
EPOCH: 6, Valid Cost: 0.468, Valid Accuracy: 0.840
EPOCH: 7, Valid Cost: 0.455, Valid Accuracy: 0.840
EPOCH: 8, Valid Cost: 0.457, Valid Accuracy: 0.840
EPOCH: 9, Valid Cost: 0.461, Valid Accuracy: 0.834
EPOCH: 10, Valid Cost: 0.445, Valid Accuracy: 0.847
EPOCH: 11, Valid Cost: 0.442, Valid Accuracy: 0.847
EPOCH: 12, Valid Cost: 0.447, Valid Accuracy: 0.849
EPOCH: 13, Valid Cost: 0.439, Valid Accuracy: 0.847
EPOCH: 14, Valid Cost: 0.442, Valid Accuracy: 0.847
EPOCH: 15, Valid Cost: 0.439, Valid Accuracy: 0.843
EPOCH: 16, Valid Cost: 0.450, Valid Accuracy: 0.846
EPOCH: 17, Valid Cost: 0.439, Valid Accuracy: 0.850
EPOCH: 18, Valid Cost: 0.444, Valid Accuracy: 0.845
EPOCH: 19, Valid Cost: 0.436, Valid Accuracy: 0.850
EPOCH: 20, Valid Cost

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [41]:
x_train, y_train, x_test = load_fashionmnist()

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

def softmax(x, axis=1):
    x = x - x.max(axis, keepdims=True)  # expのoverflowを防ぐ
    x_exp = np.exp(x)
    return x_exp / x_exp.sum(axis, keepdims=True)

# logの中身が0になるのを防ぐ
def np_log(x):
    return np.log(np.clip(a=x, a_min=1e-10, a_max=1e+10))

# 重み
W = np.random.uniform(low=-0.08, high=0.08, size=(784, 10)).astype('float32')  # 重み: (784, 10)
b = np.zeros(shape=(10,)).astype('float32')

# 学習データと検証データに分割
x_train, x_valid, y_train, y_valid = train_test_split(x_train, y_train, test_size=0.1)

def train(x, t, eps=1.0):
    """
    :param x: np.ndarray, 入力データ, (batch_size, 入力の次元数)
    :param t: np.ndarray, 教師ラベル, (batch_size, 出力の次元数)
    :param eps: float, 学習率
    """
    global W, b

    batch_size = x.shape[0]

    # 予測
    y_hat = softmax(np.matmul(x, W) + b)  # (batch_size, 出力の次元数)

    # 目的関数の評価
    cost = (- t * np_log(y_hat)).sum(axis=1).mean()
    delta = y_hat - t  # (batch_size, 出力の次元数)

    # パラメータの更新
    dW = np.matmul(x.T, delta) / batch_size  # (入力の次元数, 出力の次元数)
    db = np.matmul(np.ones(shape=(batch_size,)), delta) / batch_size  # (出力の次元数,)
    W -= eps * dW
    b -= eps * db

    return cost

def valid(x, t):
    y_hat = softmax(np.matmul(x, W) + b)
    cost = (- t * np_log(y_hat)).sum(axis=1).mean()

    return cost, y_hat

batch_size = 100
for epoch in range(20):
    # 学習データをシャッフルしてミニバッチ学習を行う
    idx = np.random.permutation(len(x_train))
    for i in range(0, len(x_train), batch_size):
        batch = idx[i:i + batch_size]
        cost = train(x_train[batch], y_train[batch], eps=0.1)

    cost, y_pred = valid(x_valid, y_valid)
    print('EPOCH: {}, Valid Cost: {:.3f}, Valid Accuracy: {:.3f}'.format(
        epoch + 1,
        cost,
        accuracy_score(y_valid.argmax(axis=1), y_pred.argmax(axis=1))
    ))
    cost, y_pred = valid(x_valid, y_valid)
    print('EPOCH: {}, Valid Cost: {:.3f}, Valid Accuracy: {:.3f}'.format(
        epoch + 1,
        cost,
        accuracy_score(y_valid.argmax(axis=1), y_pred.argmax(axis=1))
    ))
y_pred = softmax(np.matmul(x_test, W) + b)   # ← この行を追加（行頭に字下げなし）

submission = pd.Series(y_pred.argmax(axis=1), name='label')
submission.to_csv(work_dir + '/Lecture02/submission_pred_02.csv', header=True, index_label='id')

from google.colab import files
files.download(work_dir + '/Lecture02/submission_pred_02.csv')

EPOCH: 1, Valid Cost: 0.566, Valid Accuracy: 0.810
EPOCH: 1, Valid Cost: 0.566, Valid Accuracy: 0.810
EPOCH: 2, Valid Cost: 0.514, Valid Accuracy: 0.824
EPOCH: 2, Valid Cost: 0.514, Valid Accuracy: 0.824
EPOCH: 3, Valid Cost: 0.475, Valid Accuracy: 0.833
EPOCH: 3, Valid Cost: 0.475, Valid Accuracy: 0.833
EPOCH: 4, Valid Cost: 0.465, Valid Accuracy: 0.839
EPOCH: 4, Valid Cost: 0.465, Valid Accuracy: 0.839
EPOCH: 5, Valid Cost: 0.465, Valid Accuracy: 0.832
EPOCH: 5, Valid Cost: 0.465, Valid Accuracy: 0.832
EPOCH: 6, Valid Cost: 0.442, Valid Accuracy: 0.849
EPOCH: 6, Valid Cost: 0.442, Valid Accuracy: 0.849
EPOCH: 7, Valid Cost: 0.465, Valid Accuracy: 0.839
EPOCH: 7, Valid Cost: 0.465, Valid Accuracy: 0.839
EPOCH: 8, Valid Cost: 0.445, Valid Accuracy: 0.841
EPOCH: 8, Valid Cost: 0.445, Valid Accuracy: 0.841
EPOCH: 9, Valid Cost: 0.433, Valid Accuracy: 0.852
EPOCH: 9, Valid Cost: 0.433, Valid Accuracy: 0.852
EPOCH: 10, Valid Cost: 0.433, Valid Accuracy: 0.846
EPOCH: 10, Valid Cost: 0.433, 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
!ls "{work_dir}/Lecture02/data"

x_test.npy  x_train.npy  y_train.npy


In [30]:
import os
print(os.getcwd())                                    # /content であるべき
print(os.path.exists('/content/drive/MyDrive'))       # True ならマウント済み
!ls "/content/drive/MyDrive/Colab Notebooks"
!ls "/content/drive/MyDrive/Colab Notebooks/DLBasics2026_colab/Lecture02/data"

/content
True
 01_demo.ipynb
'02_knowledge_graph_embeddings.ipynb のコピー'
'02_knowledge_graph_embeddings.ipynb のコピー (1)'
'05_numpy基礎 (1).ipynb'
 05_numpy基礎.ipynb
'05_numpy基礎 (解答) (1).ipynb'
'05_numpy基礎 (解答).ipynb'
'06.04-Linear-Production-Model-in-Pyomo.ipynb のコピー'
 06_pytorch基礎.ipynb
 1208_GradientBoosting.ipynb
'1_getting_started.ipynb のコピー'
'1st_dl_autumn_Hiromitsu_Ota (1).ipynb'
 1st_dl_autumn_Hiromitsu_Ota.ipynb
'2020_0604Gym_cartpole_demo.ipynb のコピー'
'2021_0416PyTorch_CNN_demo.ipynb のコピー'
'2021_0531ccap_word2vec.ipynb のコピー'
'2021_0602nlp_demo.ipynb のコピー'
'2021_0604CLIP_demo.ipynb のコピー'
'2021_0604CLIP_demo.ipynb のコピー (1)'
'2021_0604Detectron2_demo.ipynb のコピー'
 2nd_dl_autumn_Hiromitsu_Ota.ipynb
 3rd_dl_autumn_Hiromitsu_Ota.ipynb
 3rd_MTG_G17_WorldModel_20220326.ipynb
 3行要約_試案1.ipynb
'3. 関数演習_202112281507.ipynb'
'3. 関数演習_202112281507.ipynb のコピー'
'5_summarization (1).ipynb'
'5_summarization (2).ipynb'
'5_summarization (3).ipynb'
'5_summarization (4).ipynb'
 5_summarization.

In [18]:
!ls -R "/content/drive/MyDrive/Colab Notebooks/DLBasics2026_colab" | head -50
!find /content/drive/MyDrive -name "x_train.npy" 2>/dev/null

/content/drive/MyDrive/Colab Notebooks/DLBasics2026_colab:
/content/drive/MyDrive/■松尾研_DL基礎_2026_E資格_20261009-1225/2.松尾研_DL基礎_E資格_2_20261016/1.講義資料_松尾研_DL基礎_E資格_2_20261016/x_train.npy
/content/drive/MyDrive/■松尾研_DL基礎_2026_E資格_20261009-1225/2.松尾研_DL基礎_E資格_2_20261016/1.講義資料_松尾研_DL基礎_E資格_2_20261016/Lecture02/data/x_train.npy
^C
